# 민감 키워드 매칭도 — 평판 위험도 구성 요소

## 전체 위험도 프레임워크에서의 위치
```
최종 위험도 (ABCDF)
├── 평판 위험도 33%   댓글 감성 지수 + [민감 키워드 매칭도 ← 현재]
├── 트래픽 위험도 33% 이탈 확률 + 정체 확률 + 조회수 변동성 + 업로드 주기 불확실성
└── 팬덤 위험도 33%   활성 시청자 비율 + 시청 유지 안정성
```

## 데이터 기반
재생목록 제목 컬럼이 수집되지 않아 **`video_title` 기반**으로 분석합니다.  
영상 제목은 재생목록 제목과 마찬가지로 채널의 콘텐츠 방향성을 반영합니다.

## 출력
각 채널에 대해 **민감 키워드 점수 0 ~ 1** 을 출력합니다.  
1에 가까울수록 민감 콘텐츠 비중이 높아 평판 위험이 큽니다.


## 0. 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings("ignore")

print("라이브러리 로드 완료")


## 1. 데이터 로드

In [ ]:
long = pd.read_csv("dataset_long.csv")
long["video_title"] = long["video_title"].fillna("")

print(f"long shape  : {long.shape}")
print(f"채널 수     : {long['channel_id'].nunique()}")
print(f"영상 제목 수 : {long['video_title'].notna().sum()}")


## 2. 민감 키워드 사전 정의

### 카테고리 구성 (4개 대분류)
| 대분류 | 설명 |
|---|---|
| 정치/사회 갈등 | 탄핵·내란·시위 등 직접 갈등 유발 표현 |
| 혐오/차별 | 특정 집단을 향한 혐오·차별 표현 |
| 자극/어그로성 | 폭로·충격·논란 등 자극적 콘텐츠 |
| 성인/도박/불법 | 19금·도박·마약 등 명백한 규제 대상 |

### 가중치 설계 (위험도 차등)
| 가중치 | 의미 | 예시 |
|---|---|---|
| 3점 | 고위험 — 직접적 갈등·혐오·불법 표현 | 탄핵, 혐오, 19금, 마약 |
| 2점 | 중위험 — 정치적 편향·자극성 있음 | 대통령, 충격, 논란 |
| 1점 | 저위험 — 맥락에 따라 민감할 수 있음 | 갈등, 싸움, 고소 |
| −1점 | 오탐 감점 — 무관한 정상 맥락 | 성인병, 불법주차 |


In [ ]:
KEYWORD_DICT = {
    # ── 정치/사회 갈등 ──────────────────────────────────────────────────────
    "정치사회_고위험": {
        "keywords": ["탄핵", "내란", "계엄", "시위", "집회", "파업", "폭동", "쿠데타"],
        "weight"  : 3,
        "category": "정치/사회",
    },
    "정치사회_중위험": {
        "keywords": ["정치", "국힘", "민주당", "대통령", "검찰", "여당", "야당",
                     "윤석열", "이재명", "한동훈"],
        "weight"  : 2,
        "category": "정치/사회",
    },

    # ── 혐오/차별 ───────────────────────────────────────────────────────────
    "혐오차별_고위험": {
        "keywords": ["혐오", "차별", "한남", "김치녀", "여혐", "남혐", "인종차별",
                     "틀딱", "노인충", "맘충"],
        "weight"  : 3,
        "category": "혐오/차별",
    },

    # ── 자극/어그로 ─────────────────────────────────────────────────────────
    "자극_고위험": {
        "keywords": ["폭로", "사기", "저격", "고소", "욕설", "쓰레기", "개XX",
                     "협박", "성희롱", "괴롭힘"],
        "weight"  : 3,
        "category": "자극/어그로",
    },
    "자극_중위험": {
        "keywords": ["충격", "논란", "갈등", "싸움", "미친", "어그로", "폭행",
                     "난리", "분노", "激"],
        "weight"  : 2,
        "category": "자극/어그로",
    },
    "자극_저위험": {
        "keywords": ["자극", "강렬", "도발", "파격", "위험"],
        "weight"  : 1,
        "category": "자극/어그로",
    },

    # ── 성인/도박/불법 ──────────────────────────────────────────────────────
    "성인도박불법_고위험": {
        "keywords": ["19금", "도박", "카지노", "마약", "음란", "불법도박",
                     "베팅", "토토", "성인방송"],
        "weight"  : 3,
        "category": "성인/도박/불법",
    },

    # ── 오탐 감점 (정상 맥락) ───────────────────────────────────────────────
    "오탐_감점": {
        "keywords": ["성인병", "성인식", "성인용품샵", "불법주차", "불법건축",
                     "갈등해결", "충격요법", "논란없는"],
        "weight"  : -1,
        "category": "오탐",
    },
}

# 전체 키워드 수 확인
total_kw = sum(len(v["keywords"]) for k, v in KEYWORD_DICT.items() if v["weight"] > 0)
print(f"민감 키워드 총 {total_kw}개 정의 완료")
print()
for name, info in KEYWORD_DICT.items():
    flag = "⚠ 오탐감점" if info["weight"] < 0 else f"가중치 {info['weight']}점"
    print(f"  [{info['category']:10s}] {name:20s} {flag} | {info['keywords']}")


## 3. 채널별 민감 키워드 점수 계산

### 점수 계산 로직

```
영상별 raw_score = Σ (매칭된 키워드의 가중치)
채널 raw_score   = Σ (모든 영상의 raw_score)
정규화           = raw_score / (영상 수 × 최대가중치3) → 0~1 클리핑
```

단순 매칭 수 대신 **영상 수로 나눠 정규화**하는 이유:  
영상이 많은 채널이 무조건 높은 점수를 받지 않도록 합니다.


In [ ]:
def calc_sensitive_score(group):
    titles      = group["video_title"].tolist()
    n_videos    = len(titles)
    channel_raw = 0
    cat_counts  = {cat: 0 for cat in ["정치/사회","혐오/차별","자극/어그로","성인/도박/불법"]}
    matched_titles = []

    for title in titles:
        video_score = 0
        video_matches = []

        for group_name, info in KEYWORD_DICT.items():
            for kw in info["keywords"]:
                if kw in str(title):
                    video_score += info["weight"]
                    if info["weight"] > 0:
                        video_matches.append((kw, info["category"], info["weight"]))
                        cat_counts[info["category"]] = cat_counts.get(info["category"], 0) + 1

        if video_score > 0:
            matched_titles.append({
                "title"  : title,
                "score"  : video_score,
                "matches": video_matches,
            })
        channel_raw += max(video_score, 0)   # 영상별 음수는 0으로 클램프

    # 정규화: 채널 내 영상당 평균 점수 / 최대 가중치(3)
    normalized = channel_raw / (n_videos * 3 + 1e-9)
    normalized = float(np.clip(normalized, 0, 1))

    return pd.Series({
        "raw_score"             : channel_raw,
        "sensitive_score"       : round(normalized, 4),
        "n_sensitive_videos"    : len(matched_titles),
        "sensitive_video_ratio" : round(len(matched_titles) / (n_videos + 1e-9), 4),
        "cat_politics"          : cat_counts.get("정치/사회", 0),
        "cat_hate"              : cat_counts.get("혐오/차별", 0),
        "cat_aggro"             : cat_counts.get("자극/어그로", 0),
        "cat_adult_illegal"     : cat_counts.get("성인/도박/불법", 0),
        "top_matched_titles"    : str([m["title"][:30] for m in matched_titles[:3]]),
    })

score_df = long.groupby("channel_id").apply(
    calc_sensitive_score, include_groups=False
).reset_index()

# 채널 제목 붙이기
titles_map = long.groupby("channel_id")["channel_title"].first().reset_index()
score_df   = score_df.merge(titles_map, on="channel_id")

print(f"채널 수: {len(score_df)}")
print()
print("=== sensitive_score 분포 ===")
print(score_df["sensitive_score"].describe().round(4))


## 4. 결과 확인

In [ ]:
result = score_df[[
    "channel_id","channel_title","sensitive_score",
    "n_sensitive_videos","sensitive_video_ratio",
    "cat_politics","cat_hate","cat_aggro","cat_adult_illegal",
    "top_matched_titles"
]].sort_values("sensitive_score", ascending=False).reset_index(drop=True)

print("=== 민감 키워드 점수 상위 채널 ===")
print(result[["channel_title","sensitive_score","n_sensitive_videos",
              "cat_politics","cat_aggro","top_matched_titles"]].head(10).to_string(index=False))


## 5. 시각화

In [ ]:
# ── 채널별 민감 점수 바차트 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# 왼쪽: 전체 채널 민감 점수 (내림차순)
colors = ["#D85A30" if v >= 0.1 else "#5DCAA5" if v < 0.03 else "#EF9F27"
          for v in result["sensitive_score"]]
axes[0].barh(result["channel_title"], result["sensitive_score"],
             color=colors, edgecolor="none")
for i, v in enumerate(result["sensitive_score"]):
    if v > 0:
        axes[0].text(v + 0.001, i, f"{v:.3f}", va="center", fontsize=7)
axes[0].axvline(0.1, color="#D85A30", linestyle="--", linewidth=1, alpha=0.6, label="고위험 기준(0.1)")
axes[0].axvline(0.03, color="#EF9F27", linestyle="--", linewidth=1, alpha=0.6, label="중위험 기준(0.03)")
axes[0].set_xlabel("민감 키워드 점수 (0~1)")
axes[0].set_title("채널별 민감 키워드 점수\n빨강≥0.1 고위험  주황≥0.03 중위험  초록 안전")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="y", labelsize=8)

# 오른쪽: 카테고리별 스택 바차트 (상위 15개)
top15 = result.head(15).sort_values("sensitive_score")
cat_cols = ["cat_politics","cat_hate","cat_aggro","cat_adult_illegal"]
cat_labels = ["정치/사회","혐오/차별","자극/어그로","성인/도박/불법"]
cat_colors = ["#378ADD","#D85A30","#EF9F27","#7F77DD"]

left = np.zeros(len(top15))
for col, label, color in zip(cat_cols, cat_labels, cat_colors):
    vals = top15[col].values
    axes[1].barh(top15["channel_title"], vals, left=left,
                 label=label, color=color, edgecolor="none", alpha=0.85)
    left += vals

axes[1].set_xlabel("민감 키워드 매칭 횟수")
axes[1].set_title("카테고리별 민감 키워드 분포 (상위 15개 채널)")
axes[1].legend(fontsize=8, loc="lower right")
axes[1].tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# 점수 분포 히스토그램 + 위험도 구간
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(result["sensitive_score"], bins=15,
        color="#378ADD", edgecolor="white", alpha=0.8)
ax.axvline(0.03, color="#EF9F27", linestyle="--", lw=1.5, label="중위험(0.03)")
ax.axvline(0.10, color="#D85A30", linestyle="--", lw=1.5, label="고위험(0.10)")
ax.set_xlabel("민감 키워드 점수 (0~1)")
ax.set_ylabel("채널 수")
ax.set_title("민감 키워드 점수 분포")
ax.legend()
plt.tight_layout()
plt.show()

# 위험 구간별 채널 수
low    = (result["sensitive_score"] <  0.03).sum()
mid    = ((result["sensitive_score"] >= 0.03) & (result["sensitive_score"] < 0.10)).sum()
high   = (result["sensitive_score"] >= 0.10).sum()
print(f"안전  (0.00~0.03): {low}개 채널")
print(f"중위험(0.03~0.10): {mid}개 채널")
print(f"고위험(0.10~1.00): {high}개 채널")


## 6. 저장

In [ ]:
output = result[[
    "channel_id","channel_title",
    "sensitive_score","n_sensitive_videos","sensitive_video_ratio",
    "cat_politics","cat_hate","cat_aggro","cat_adult_illegal",
]].copy()

output.to_csv("sensitive_keyword_score.csv", index=False, encoding="utf-8-sig")
print("저장 완료: sensitive_keyword_score.csv")
print()
print("=== 최종 출력 샘플 ===")
print(output[["channel_title","sensitive_score","n_sensitive_videos"]].to_string(index=False))


## 점수 해석 가이드

| 점수 구간 | 위험도 | 의미 |
|---|---|---|
| 0.00 ~ 0.03 | 안전 | 민감 콘텐츠 거의 없음 |
| 0.03 ~ 0.10 | 중위험 | 일부 자극적·정치적 콘텐츠 포함 |
| 0.10 ~ 1.00 | 고위험 | 민감 콘텐츠 비중 높아 평판 리스크 존재 |

## 한계 및 개선 방향

| 한계 | 개선 방향 |
|---|---|
| 단순 문자열 매칭 → 맥락 무시 (예: "충격적으로 맛있는") | 댓글 감성 분석(KoBERT)과 결합 시 정확도 향상 |
| 키워드 사전이 고정됨 | 정기적 키워드 업데이트 필요 |
| 재생목록 제목 미수집 | 팀원이 수집 후 `playlist_title` 컬럼 추가 시 대체 가능 |

## 다음 통합 단계
이 점수(`sensitive_score`)는 **평판 위험도 33%** 중  
댓글 감성 지수와 함께 가중 평균으로 합산됩니다.

```python
평판_위험도 = (댓글_감성_위험점수 × 0.5) + (sensitive_score × 0.5)
```
